# ReactorCell Digital Twin

The **ReactorCell** closes the full fuselk loop:

```
ECE shot → HELIX → ELM + disruption detect → Venturi/RL actuate → KPI update
```

**FusionKPIs** composite score weights TBR, ELM-free fraction, divertor uniformity, disruption risk, muon gain, and HELIX SNR.

---

## Composite fusion functional (cross-domain score)

$$\mathcal{F} = 0.25\, f_{TBR} + 0.20\, f_{ELM} + 0.15\, f_{div} + 0.20\, (1 - P_{dis}) + 0.10\, f_\mu + 0.10\, f_{SNR}$$

where each term is normalized to $[0,1]$. This is the **abstract scalar** that couples plasma control, fuel cycle, and muon catalysis into one benchmark — no other open toolchain defines such a functional.

`FusionCell` extends `ReactorCell` with full trifecta stripping + brine coating hypothesis.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "src" / "deepiri_fuselk").exists() and (repo.parent / "src" / "deepiri_fuselk").exists():
    repo = repo.parent

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]


In [ ]:
from deepiri_fuselk.sim import ReactorCell
from deepiri_fuselk.sim.fusion_kpis import FusionKPIs

cell = ReactorCell(grid_size=24, train_elm=True)
run = cell.run(n_steps=40, seed=42)
report = run.to_report()

print("Reactor run report:")
for k, v in report.items():
    print(f"  {k}: {v}")


In [ ]:
steps = [s.step for s in run.steps]
scores = [s.kpis.score() for s in run.steps]
tbr = [s.kpis.tritium_breeding_ratio for s in run.steps]
elm_free = [s.kpis.elm_free_fraction for s in run.steps]
dis_risk = [s.kpis.disruption_risk for s in run.steps]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].plot(steps, scores, "k-", lw=2)
axes[0, 0].set_ylabel("Fusion score"); axes[0, 0].set_title("Composite KPI trajectory")

axes[0, 1].plot(steps, tbr, label="TBR")
axes[0, 1].axhline(1.05, color="gray", ls="--")
axes[0, 1].set_ylabel("TBR"); axes[0, 1].legend()

axes[1, 0].plot(steps, elm_free, color="C2", label="ELM-free fraction")
axes[1, 0].plot(steps, [1 - d for d in dis_risk], color="C3", label="1 - disruption risk")
axes[1, 0].set_xlabel("step"); axes[1, 0].legend()

last = run.steps[-1]
hf = last.heat_flux
if hf.ndim == 1:
    side = int(np.sqrt(hf.size))
    hf = hf[: side * side].reshape(side, side)
axes[1, 1].imshow(hf, cmap="hot", origin="lower")
axes[1, 1].set_title(f"Final heat flux (action: {last.action_taken})")

plt.tight_layout()
plt.show()


## DigitalTwin comparison

In [ ]:
from deepiri_fuselk.sim import DigitalTwin

twin = DigitalTwin(grid_size=16)
twin.reset()
for _ in range(30):
    twin.step()
summary = twin.summary()

print("DigitalTwin summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")


## FusionKPI score decomposition (last step)

In [ ]:
k = run.steps[-1].kpis
components = {
    "TBR (25%)": min(1.0, k.tritium_breeding_ratio / 1.05),
    "ELM-free (20%)": k.elm_free_fraction,
    "Divertor (15%)": k.divertor_uniformity,
    "Disruption (20%)": 1.0 - k.disruption_risk,
    "Muon (10%)": min(1.0, k.muon_gain / 284.0),
    "HELIX SNR (10%)": min(1.0, k.helix_snr_mean / 5.0),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(components.keys()), list(components.values()), color="steelblue")
ax.set_xlim(0, 1); ax.set_xlabel("normalized contribution")
ax.set_title(f"KPI breakdown — total score = {k.score():.3f}")
plt.tight_layout()
plt.show()


## FusionCell — full VISION architecture

In [ ]:
from deepiri_fuselk.sim.fusion_cell import FusionCell

fusion = FusionCell(grid_size=20, train_elm=False)
run, report = fusion.run(n_steps=25, seed=7)

print(f"Fusion score: {report.fusion_score:.3f}")
print(f"TBR={report.fuel_cycle.tritium_breeding_ratio:.3f}, Pe={report.fuel_cycle.peclet_number:.2f}, extraction_ok={report.fuel_cycle.extraction_ok}")
print(f"Muon fusions/μ={report.muon_cycle.fusions_per_muon:.1f}, breakeven={report.muon_cycle.breakeven}")
print(f"Brine viable={report.fuel_cycle.brine_viable}, heat reduction={report.fuel_cycle.brine_heat_reduction:.2f}")
print(f"Actions taken: {report.recommended_actions}")
